# درس ۰۲ - بررسی چارچوب عامل مایکروسافت

**چارچوب عامل مایکروسافت (MAF)** یک چارچوب یکپارچه برای ساخت عوامل هوش مصنوعی است. این چارچوب معماری تمیز و ترکیبی با چهار بلوک ساختاری اصلی ارائه می‌دهد:

- **مشتری** – به نقطه پایان مدل هوش مصنوعی متصل می‌شود و مدیریت ارتباط را بر عهده دارد
- **عامل** – یک مشتری را با دستورالعمل‌ها و تعاریف ابزارها پوشش می‌دهد
- **ابزارها** – قابلیت‌های عامل را با توابع سفارشی که مدل می‌تواند فراخوانی کند گسترش می‌دهند
- **جلسه** – تاریخچه مکالمه را برای تعاملات چند نوبته حفظ می‌کند

در این درس، یک **عامل رزرو سفر** می‌سازیم که با استفاده از این مفاهیم، در دسترس بودن مقصد را بررسی می‌کند.


## تنظیمات


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## درک معماری چارچوب عامل

چارچوب عامل مایکروسافت از یک معماری لایه‌ای پیروی می‌کند:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **کلاینت** – یک `AzureAIProjectAgentProvider` به یک پیاده‌سازی Azure OpenAI متصل می‌شود. این لایه مسئول احراز هویت، قالب‌بندی درخواست‌ها و تجزیه پاسخ‌ها است.
2. **عامل** – که از طریق `provider.create_agent()` از کلاینت ایجاد می‌شود، این عامل دسترسی به مدل را با دستورالعمل‌ها (دستور سیستم) و ابزارها ترکیب می‌کند.
3. **ابزارها** – توابع پایتون که با `@tool` تزئین شده‌اند و عامل می‌تواند برای انجام عملیات یا بازیابی داده‌ها آن‌ها را فراخوانی کند.
4. **نشست** – یک شی `AgentSession` (ایجاد شده از طریق `agent.create_session()`) که سابقه مکالمه را ذخیره می‌کند و امکان گفتگو چند مرحله‌ای را فراهم می‌سازد، جایی که عامل متن قبلی را به خاطر می‌آورد.

بیایید هر لایه را گام به گام بسازیم.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## افزودن ابزارها با دکوراتور @tool

ابزارها به عامل‌ها امکان انجام اقداماتی فراتر از تولید متن می‌دهند. دکوراتور `@tool` یک تابع معمولی پایتون را به چیزی تبدیل می‌کند که عامل می‌تواند آن را فراخوانی کند.

نکات کلیدی:
- از `Annotated[type, "description"]` استفاده کنید تا مدل هر پارامتر را درک کند.
- رشته مستندسازی تبدیل به توضیح ابزار می‌شود که مدل آن را می‌بیند.
- `approval_mode="never_require"` یعنی ابزار به‌صورت خودکار بدون تأیید کاربر اجرا می‌شود.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## ایجاد یک عامل با ابزارها

حال ما کلاینت، دستورالعمل‌ها و ابزارها را در یک عامل ترکیب می‌کنیم. `instructions` به‌عنوان پرامپت سیستم عمل می‌کنند — آن‌ها شخصیت و رفتار عامل را تعریف می‌کنند.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## مکالمات چند مرحله‌ای با جلسات

یک `AgentSession` (که از طریق `agent.create_session()` ایجاد می‌شود) تمام پیام‌های یک مکالمه را پیگیری می‌کند. با ارسال همان جلسه به هر فراخوانی `agent.run()`، عامل به تمام تاریخچه مکالمه دسترسی دارد و می‌تواند به پیام‌های قبلی ارجاع دهد.

ما `tools=[check_destination_availability]` را می‌فرستیم تا عامل بتواند در هر مرحله ابزار بررسی در دسترس بودن ما را فراخوانی کند.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## خلاصه

در این درس شما با چهار ستون چارچوب Microsoft Agent آشنا شدید:

| مفهوم | آنچه یاد گرفتید |
|---------|------------------|
| **مشتری** | `AzureAIProjectAgentProvider` با احراز هویت مبتنی بر اعتبارنامه به Azure OpenAI متصل می‌شود |
| **عامل** | `provider.create_agent()` اتصال مدل را با دستورالعمل‌ها و یک نام بسته‌بندی می‌کند |
| **ابزارها** | دکوراتور `@tool` توابع پایتون را برای فراخوانی توسط عامل در دسترس قرار می‌دهد |
| **نشست** | `agent.create_session()` تاریخچه گفتگو را در چند نوبت حفظ می‌کند |

این بلوک‌های سازنده با هم ترکیب می‌شوند تا عامل‌هایی بسازند که می‌توانند مکالمات طبیعی داشته باشند، توابع خارجی را فراخوانی کنند و زمینه را حفظ کنند — پایه‌ای برای الگوهای پیشرفته‌تر عاملی در درس‌های بعدی.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**توضیح مهم**:  
این سند با استفاده از سرویس ترجمه ماشینی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما برای دقت تلاش می‌کنیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است حاوی اشتباهات یا نواقص باشند. سند اصلی به زبان مبداء باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما مسئول هیچگونه سوءتفاهم یا برداشت نادرستی ناشی از استفاده از این ترجمه نیستیم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
